In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import pandas as pd
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from src.data.augment import data_transformer
from src.utils.utils import process_dicom_to_3channel
from src.utils.utils import rle2mask
from src.models.classifier import MedicalFusionClassifier
from src.models.segmentor import build_stage2_segmentor
from configs.configs import get_config

In [ ]:

class FusionModelWrapper(torch.nn.Module):
    def __init__(self, model, metadata):
        super(FusionModelWrapper, self).__init__()
        self.model = model
        self.metadata = metadata

    def forward(self, x):
        return self.model(x, self.metadata)

def plot_evaluation_suite(df, TP_idx, classifier_model, transform_pipeline, device="cuda"):
    """
    Plots a 1x4 grid for each index: Original, Grad-CAM, True Mask, Pred Mask.
    """
    classifier_model.to(device)
    classifier_model.eval()
    print(f"Total indices to process: {len(TP_idx)}")
    
    for count, i in enumerate(TP_idx): 
        print(f"Processing item {count+1}/{len(TP_idx)} (DataFrame Index: {i})...")
        
        # try:
        row = df.iloc[i]
        dicom_path = row['path']
        true_mask_raw = row['EncodedPixels']
        pred_mask_raw = row['pred_mask']

        # Skip if masks are missing
        if pd.isna(true_mask_raw) or pd.isna(pred_mask_raw):
            print(f"⚠️ Skipping index {i}: One of the masks is NaN/empty.")
            continue

        # ---------------------------------------------------------
        # 1. Base Image Processing (1024x1024)
        # ---------------------------------------------------------
        raw_rgb = process_dicom_to_3channel(dicom_path) # Returns uint8 RGB
        scaled_rgb = raw_rgb.astype(np.float32) / 255.0   # Scale for Grad-CAM visualizer
        
        # ---------------------------------------------------------
        # 2. Prepare Tensors for the Classifier
        # ---------------------------------------------------------
        meta_values = row[['Age', 'Sex', 'ViewPosition']].astype(float).values
        metadata_tensor = torch.tensor(meta_values, dtype=torch.float32).unsqueeze(0).to(device)
        
        augmented = transform_pipeline(image=raw_rgb)
        image_tensor = augmented['image'].unsqueeze(0).to(device)
        
        # ADD THIS LINE: Slice the 3-channel tensor down to 1-channel (Keep Batch, 1 Channel, H, W)
        image_tensor = image_tensor[:, :1, :, :] 
        
        # ---------------------------------------------------------
        # 3. Generate Grad-CAM Heatmap
        # ---------------------------------------------------------
        wrapped_model = FusionModelWrapper(classifier_model, metadata_tensor)
        
        # ⚠️ UPDATE THIS TARGET LAYER BASED ON YOUR BACKBONE:
        # For ResNet34: [classifier_model.image_encoder.layer4[-1]]
        # For EfficientNet: [classifier_model.image_encoder.conv_head]
        target_layers = [classifier_model.image_encoder.conv_head]
        
        cam = GradCAM(model=wrapped_model, target_layers=target_layers)
        targets = [ClassifierOutputTarget(0)] # Target the positive logit (Index 0)
        
        # Generate the raw grayscale mask (output matches image_tensor shape, e.g., 512x512)
        grayscale_cam = cam(input_tensor=image_tensor, targets=targets)[0, :]
        
        # Resize the heatmap back up to 1024x1024 to match the original image and masks
        grayscale_cam_resized = cv2.resize(grayscale_cam, (raw_rgb.shape[1], raw_rgb.shape[0]))
        
        # Overlay heatmap on the normalized RGB image
        heatmap_overlay = show_cam_on_image(scaled_rgb, grayscale_cam_resized, use_rgb=True)

        # ---------------------------------------------------------
        # 4. Mask Decoding & Formatting
        # ---------------------------------------------------------
        true_mask = rle2mask(true_mask_raw, 1024, 1024)
        true_mask = np.rot90(true_mask, 3) 
        true_mask = np.flip(true_mask, axis=1)
        
        pred_mask = rle2mask(pred_mask_raw, 1024, 1024)
        pred_mask = np.rot90(pred_mask, 3) 
        pred_mask = np.flip(pred_mask, axis=1)
        
        true_mask_visible = np.ma.masked_where(true_mask == 0, true_mask)
        pred_mask_visible = np.ma.masked_where(pred_mask == 0, pred_mask)
        
        # ---------------------------------------------------------
        # 5. Plot the 1x4 Grid
        # ---------------------------------------------------------
        fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 5))
        
        axes[0].imshow(raw_rgb)
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        axes[1].imshow(heatmap_overlay)
        axes[1].set_title("Classifier Grad-CAM")
        axes[1].axis('off')
        
        axes[2].imshow(raw_rgb)
        axes[2].imshow(true_mask_visible, cmap='Reds', alpha=0.5, vmin=0, vmax=1) 
        axes[2].set_title("True Mask Overlay")
        axes[2].axis('off')  
        
        axes[3].imshow(raw_rgb)
        axes[3].imshow(pred_mask_visible, cmap='Blues', alpha=0.5, vmin=0, vmax=1)
        axes[3].set_title("Predicted Mask Overlay")
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()
        plt.close(fig) # Prevent memory leaks
        
        # except Exception as e:
        #     print(f"❌ Error at index {i}: {e}")
        #     continue

In [ ]:
cfg = get_config()

device = torch.device(cfg.device)
classifier_model = MedicalFusionClassifier(
    backbone_name=cfg.model.classifier_backbone, 
    num_meta_features=cfg.model.num_meta_features
).to(device)
classifier_model.load_state_dict(
    torch.load(cfg.model_paths.classifier_checkpoint, map_location=device)
)

segmentor_model = build_stage2_segmentor().to(device) # If build_stage2_segmentor takes args, use cfg.model here too
segmentor_model.load_state_dict(
    torch.load(cfg.model_paths.segmentor_checkpoint, map_location=device)
)
# Create your validation transform (e.g., Resize to 512 + Normalize + ToTensor)
val_transforms = data_transformer(phase='val', size=1024)
val_df = pd.read_csv('val_data_with_prediction.csv')
val_df = val_df[(val_df['class'] == 1) & (val_df['pred'] == 1)].reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Execute the visualization suite
plot_evaluation_suite(
    df=val_df, 
    TP_idx=val_df.index, 
    classifier_model=classifier_model, 
    transform_pipeline=val_transforms,
    device=device
)